[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/02_text_chunking/02_text_chunking_solutions.ipynb)

# 02. 텍스트 청킹 & PDF 처리 실습 — 연습 문제 해설

[02_text_chunking.ipynb](02_text_chunking.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q langchain-text-splitters langchain-core tiktoken

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import tiktoken

sample_text = (
    "제1조(목적) 이 규정은 임직원의 휴가 사용에 관한 사항을 정함을 목적으로 한다.\n\n제2조(연차휴가) 1년간 80퍼센트 이상 출근한 임직원에게는 15일의 연차휴가를 부여한다. 3년 이상 계속 근로한 임직원에 대하여는 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 유급휴가를 부여한다.\n\n제3조(육아휴직) 임신 중인 여성 임직원이나 만 8세 이하 자녀를 양육하는 임직원은 최대 1년의 육아휴직을 신청할 수 있다. 육아휴직 기간은 근속연수에 포함하되, 승급을 위한 근속기간에는 산입하지 아니한다.\n\n제4조(경조휴가) 결혼, 출산, 사망 등 경조사가 발생한 경우 별표에서 정한 일수의 유급휴가를 부여한다. 본인 결혼은 5일, 배우자 출산은 10일, 부모 사망은 5일로 한다.\n\n제5조(병가) 업무 외 질병이나 부상으로 근로가 불가능한 경우 연간 60일 이내에서 병가를 사용할 수 있다. 병가 3일 이상 사용 시 진단서를 제출하여야 한다.\n\n제6조(휴가의 신청) 휴가를 사용하고자 하는 임직원은 사용 예정일 3일 전까지 소속 부서장에게 신청하여야 한다. 다만 병가 등 부득이한 사유가 있는 경우에는 사후에 신청할 수 있다."
)
doc = Document(page_content=sample_text, metadata={"source": "휴가규정.txt"})

## 연습 1. 조항(`제N조`) 경계를 우선으로 자르기

In [ ]:
default_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=20, separators=["\n\n", "\n", ". ", " ", ""]
)
article_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=20, separators=["\n\n제", "\n\n", "\n", ". ", " ", ""]
)

default_chunks = default_splitter.split_documents([doc])
article_chunks = article_splitter.split_documents([doc])

same = [c.page_content for c in default_chunks] == [c.page_content for c in article_chunks]
print("두 결과가 동일한가?", same)

print("\n=== 기본 separators ===")
for c in default_chunks:
    print(repr(c.page_content[:40]))

print("\n=== '제N조' 우선 separators ===")
for c in article_chunks:
    print(repr(c.page_content[:40]))

**해설 1**
- 직접 실행해보면 `chunk_size=200`에서는 두 결과가 **똑같이** 나옵니다. `chunk_size`를 여러 값으로 바꿔가며 비교해봐도(직접 시도해보세요) 대부분 동일하거나, 다르더라도 원하는 방향(조항이 절대 섞이지 않음)으로 안정적으로 나오지는 않습니다.
- 이유는 `separators`의 역할을 오해하기 쉽기 때문입니다. `separators` 우선순위는 "너무 큰 텍스트 조각을 어떤 기준으로 더 잘게 쪼갤지"만 정할 뿐입니다. 일단 조각들이 `chunk_size`보다 작아지고 나면, 그 조각들을 다시 이웃끼리 합쳐서 청크를 채우는 병합(merge) 단계가 있는데, 이 병합은 **어떤 구분자로 쪼개졌는지와 무관하게 그냥 크기가 맞으면 욕심 있게(greedy) 합칩니다.** 그래서 `"\n\n제"`를 최우선 구분자로 줘도, 조항 두 개가 합쳐서 `chunk_size` 이하라면 결국 한 청크로 다시 합쳐질 수 있습니다.
- 즉 `separators` 조정만으로는 "청크가 절대 조항 경계를 넘지 않는다"를 보장할 수 없습니다.

### 확실하게 조항 경계를 지키는 방법: 정규식으로 먼저 분리하기

조항 경계를 100% 보장하려면, `RecursiveCharacterTextSplitter`에만 맡기지 말고 **정규식으로 먼저 조항 단위로 자른 뒤**, 개별 조항이 `chunk_size`보다 길 때만 그 조항 안에서 추가로 청킹합니다.

In [ ]:
import re

def split_by_article(text: str, chunk_size: int = 200, chunk_overlap: int = 20) -> list[str]:
    # '제N조' 바로 앞에서 자른다 (lookahead라서 '제N조' 자체는 잘려나가지 않고 다음 조각의 시작이 된다)
    articles = [a.strip() for a in re.split(r"(?=제\d+조)", text) if a.strip()]

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    result = []
    for article in articles:
        if len(article) <= chunk_size:
            result.append(article)  # 조항 하나가 충분히 작으면 그대로 청크 하나로 유지
        else:
            result.extend(splitter.split_text(article))  # 너무 길 때만 조항 내부를 추가로 쪼갠다
    return result


article_first_chunks = split_by_article(sample_text, chunk_size=120, chunk_overlap=20)
print(f"{len(article_first_chunks)}개 청크 (어떤 청크도 조항 경계를 넘나들지 않음)")
for c in article_first_chunks:
    print("-", c[:40])

**해설 2**
- 조항별로 먼저 나누고 나면, 어떤 청크도 서로 다른 두 조항의 내용을 섞어서 담지 않는다는 것이 구조적으로 보장됩니다 — `chunk_size`를 아무리 바꿔도 조항과 조항이 한 청크에 합쳐지는 일이 없습니다.
- 대신 조항 하나가 `chunk_size`보다 길면 그 조항 하나는 여러 청크로 나뉩니다(위 예시의 제2조, 제3조처럼). 이건 어쩔 수 없는 트레이드오프이지만, 적어도 "서로 다른 조항이 한 청크에 뒤섞이는" 것보다는 검색 결과의 출처가 훨씬 명확해집니다.
- 실무에서도 문서 구조(마크다운 헤더, 조항 번호, HTML 태그 등)를 알고 있다면, `RecursiveCharacterTextSplitter` 단독보다 이렇게 **구조 인식 전처리 + 청킹을 조합**하는 방식이 훨씬 안정적입니다.

## 연습 2. 청크별 토큰 수 분포 확인

In [ ]:
encoding = tiktoken.get_encoding("cl100k_base")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
chunks = splitter.split_documents([doc])

token_counts = [len(encoding.encode(c.page_content)) for c in chunks]
char_counts = [len(c.page_content) for c in chunks]

for i, (t, ch) in enumerate(zip(token_counts, char_counts)):
    print(f"chunk {i}: 글자수={ch:>4}  토큰수={t:>4}  (토큰/글자 비율={t / ch:.2f})")

print(f"\n평균 토큰 수: {sum(token_counts) / len(token_counts):.1f}")
print(f"최대 토큰 수: {max(token_counts)}")

**해설**
- 한글 텍스트는 영어보다 글자당 토큰 소비가 커서(실습 7 참고), 토큰/글자 비율이 1에 가깝게(때로는 그 이상으로) 나옵니다. 즉 `chunk_size=500`자로 잘라도 토큰 수는 500보다 살짝 적은 정도이지, 영어 텍스트처럼 글자 수의 절반 이하로 뚝 떨어지지는 않습니다 — 직접 실행한 값으로 확인해보세요.
- 이 비율을 알아두면, OpenAI API의 컨텍스트 길이 제한(토큰 기준)에 맞춰 `chunk_size`(글자 기준)를 얼마나 크게 잡아도 안전한지 역산할 수 있습니다. 한글 문서에서 토큰/글자 비율이 1.0에 가깝다면, 컨텍스트에 4000 토큰까지 넣고 싶을 때 청크를 4000자보다 살짝 작게만 잡아도 대체로 안전합니다 — 영어 문서라면 토큰/글자 비율이 훨씬 낮으므로(실습 7의 영문 예시 참고) 같은 토큰 예산 안에서 글자 기준 `chunk_size`를 더 크게 잡을 수 있습니다.